In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.2 MB/s eta 0:00:00


In [4]:
import os
import csv
import json
import shutil
import numpy as np
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, ssd300_vgg16
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssd import SSDClassificationHead
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
from ultralytics import YOLO
import zipfile

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DRIVE_DIR     = "/content/drive/MyDrive/RD2"
MODELS_DIR    = f"{DRIVE_DIR}/models"
OUTPUT_DIR    = "/content/inference_output"
IMG_SIZE_YOLO = 640
IMG_SIZE_RCNN = 640
IMG_SIZE_SSD  = 300
NUM_CLASSES   = 3
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IOU_THRESH    = 0.1  # threshold for spatial relationship inference
CONF_THRESH   = 0.05 # confidence threshold for detections
CLASS_NAMES   = {0: "person", 1: "smartphone"}  # YOLO
CLASS_MAP_VOC = {1: "person", 2: "smartphone"}  # Faster R-CNN / SSD
# ──────────────────────────────────────────────────────────────────────────────

print(f"Using device: {DEVICE}")

# Create output directories
for model_name in ["yolo", "fasterrcnn", "ssd"]:
    os.makedirs(f"{OUTPUT_DIR}/{model_name}/annotated", exist_ok=True)

# ─── EXTRACT DATASET ──────────────────────────────────────────────────────────
os.makedirs("/content/dataset/yolo", exist_ok=True)
with zipfile.ZipFile(f"{DRIVE_DIR}/yolo_dataset.zip", "r") as z:
    z.extractall("/content/dataset/yolo")

IMAGE_DIR = "/content/dataset/yolo/yolo/Smartphone-Detection-3/train/images"
image_files = sorted([
    str(f) for f in Path(IMAGE_DIR).iterdir()
    if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
])
print(f"Total images: {len(image_files)}")

# ─── SPATIAL RELATIONSHIP: IS PHONE NEAR PERSON? ──────────────────────────────
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def is_phone_near_person(person_boxes, phone_boxes, iou_thresh=IOU_THRESH):
    """Returns True if any phone box overlaps with any person box."""
    for pb in phone_boxes:
        for prb in person_boxes:
            if compute_iou(pb, prb) > iou_thresh:
                return True
            # Also check if phone centre is inside person box
            cx = (pb[0] + pb[2]) / 2
            cy = (pb[1] + pb[3]) / 2
            if prb[0] <= cx <= prb[2] and prb[1] <= cy <= prb[3]:
                return True
    return False

# ─── DRAWING HELPER ───────────────────────────────────────────────────────────
def draw_results(img_path, person_boxes, phone_boxes, usage_detected, model_name):
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
        small_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 16)
    except:
        font = ImageFont.load_default()
        small_font = font

    # Draw person boxes in blue
    for box in person_boxes:
        draw.rectangle(box, outline="blue", width=3)
        draw.text((box[0]+4, box[1]+4), "person", fill="blue", font=small_font)

    # Draw phone boxes in red
    for box in phone_boxes:
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0]+4, box[1]+4), "smartphone", fill="red", font=small_font)

    # Draw usage banner at top
    banner_color = (0, 180, 0) if usage_detected else (200, 0, 0)
    banner_text  = "✅ Smartphone Usage Detected" if usage_detected else "❌ No Usage Detected"
    draw.rectangle([0, 0, img.width, 40], fill=banner_color)
    draw.text((10, 8), banner_text, fill="white", font=font)

    # Save
    out_path = f"{OUTPUT_DIR}/{model_name}/annotated/{Path(img_path).name}"
    img.save(out_path)
    return out_path

# ─── RESCALE BOXES TO ORIGINAL IMAGE SIZE ─────────────────────────────────────
def rescale_boxes(boxes, orig_w, orig_h, model_size):
    scale_x = orig_w / model_size
    scale_y = orig_h / model_size
    rescaled = []
    for box in boxes:
        rescaled.append([
            box[0] * scale_x,
            box[1] * scale_y,
            box[2] * scale_x,
            box[3] * scale_y
        ])
    return rescaled

# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 1: YOLOv8
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*50}")
print("RUNNING YOLOV8 INFERENCE")
print(f"{'='*50}")

yolo_model = YOLO(f"{MODELS_DIR}/yolo/yolo_fold1_best.pt")
yolo_results = []

for img_path in image_files:
    img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img.size

    results = yolo_model(img_path, verbose=False, conf=CONF_THRESH)
    boxes   = results[0].boxes

    person_boxes = []
    phone_boxes  = []

    for box in boxes:
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        if cls_id == 0:
            person_boxes.append([x1, y1, x2, y2])
        elif cls_id == 1:
            phone_boxes.append([x1, y1, x2, y2])

    usage = is_phone_near_person(person_boxes, phone_boxes)
    draw_results(img_path, person_boxes, phone_boxes, usage, "yolo")

    yolo_results.append({
        "image":           Path(img_path).name,
        "persons":         len(person_boxes),
        "phones":          len(phone_boxes),
        "usage_detected":  usage
    })

# Save CSV
with open(f"{OUTPUT_DIR}/yolo/results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image", "persons", "phones", "usage_detected"])
    writer.writeheader()
    writer.writerows(yolo_results)

yolo_usage_count = sum(1 for r in yolo_results if r["usage_detected"])
print(f"YOLOv8 — Usage detected in {yolo_usage_count}/{len(image_files)} images")

# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 2: Faster R-CNN
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*50}")
print("RUNNING FASTER R-CNN INFERENCE")
print(f"{'='*50}")

rcnn_model = fasterrcnn_resnet50_fpn(weights=None)
in_features = rcnn_model.roi_heads.box_predictor.cls_score.in_features
rcnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
rcnn_model.load_state_dict(torch.load(
    f"{MODELS_DIR}/fasterrcnn/fasterrcnn_fold5.pt",
    map_location=DEVICE
))
rcnn_model.to(DEVICE)
rcnn_model.eval()

rcnn_results = []

for img_path in image_files:
    img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img.size
    img_resized = img.resize((IMG_SIZE_RCNN, IMG_SIZE_RCNN))
    img_tensor  = transforms.ToTensor()(img_resized).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = rcnn_model(img_tensor)[0]

    pred_boxes  = output["boxes"].cpu().numpy()
    pred_scores = output["scores"].cpu().numpy()
    pred_labels = output["labels"].cpu().numpy()

    keep = pred_scores >= CONF_THRESH
    pred_boxes  = pred_boxes[keep]
    pred_labels = pred_labels[keep]

    person_boxes = []
    phone_boxes  = []

    for box, label in zip(pred_boxes, pred_labels):
        rescaled = rescale_boxes([box], orig_w, orig_h, IMG_SIZE_RCNN)[0]
        if label == 1:
            person_boxes.append(rescaled)
        elif label == 2:
            phone_boxes.append(rescaled)

    usage = is_phone_near_person(person_boxes, phone_boxes)
    draw_results(img_path, person_boxes, phone_boxes, usage, "fasterrcnn")

    rcnn_results.append({
        "image":          Path(img_path).name,
        "persons":        len(person_boxes),
        "phones":         len(phone_boxes),
        "usage_detected": usage
    })

# Save CSV
with open(f"{OUTPUT_DIR}/fasterrcnn/results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image", "persons", "phones", "usage_detected"])
    writer.writeheader()
    writer.writerows(rcnn_results)

rcnn_usage_count = sum(1 for r in rcnn_results if r["usage_detected"])
print(f"Faster R-CNN — Usage detected in {rcnn_usage_count}/{len(image_files)} images")

# ═══════════════════════════════════════════════════════════════════════════════
# MODEL 3: SSD
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*50}")
print("RUNNING SSD INFERENCE")
print(f"{'='*50}")

ssd_model = ssd300_vgg16(weights=None)
num_anchors = ssd_model.anchor_generator.num_anchors_per_location()
in_channels = [512, 1024, 512, 256, 256, 256]
ssd_model.head.classification_head = SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=NUM_CLASSES
)
ssd_model.load_state_dict(torch.load(
    f"{MODELS_DIR}/ssd/ssd_fold2.pt",
    map_location=DEVICE
))
ssd_model.to(DEVICE)
ssd_model.eval()

ssd_results = []

for img_path in image_files:
    img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img.size
    img_resized = img.resize((IMG_SIZE_SSD, IMG_SIZE_SSD))
    img_tensor  = transforms.ToTensor()(img_resized).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = ssd_model(img_tensor)[0]

    pred_boxes  = output["boxes"].cpu().numpy()
    pred_scores = output["scores"].cpu().numpy()
    pred_labels = output["labels"].cpu().numpy()

    keep = pred_scores >= CONF_THRESH
    pred_boxes  = pred_boxes[keep]
    pred_labels = pred_labels[keep]

    person_boxes = []
    phone_boxes  = []

    for box, label in zip(pred_boxes, pred_labels):
        rescaled = rescale_boxes([box], orig_w, orig_h, IMG_SIZE_SSD)[0]
        if label == 1:
            person_boxes.append(rescaled)
        elif label == 2:
            phone_boxes.append(rescaled)

    usage = is_phone_near_person(person_boxes, phone_boxes)
    draw_results(img_path, person_boxes, phone_boxes, usage, "ssd")

    ssd_results.append({
        "image":          Path(img_path).name,
        "persons":        len(person_boxes),
        "phones":         len(phone_boxes),
        "usage_detected": usage
    })

# Save CSV
with open(f"{OUTPUT_DIR}/ssd/results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image", "persons", "phones", "usage_detected"])
    writer.writeheader()
    writer.writerows(ssd_results)

ssd_usage_count = sum(1 for r in ssd_results if r["usage_detected"])
print(f"SSD — Usage detected in {ssd_usage_count}/{len(image_files)} images")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*50}")
print("SMARTPHONE USAGE DETECTION SUMMARY")
print(f"{'='*50}")
print(f"Total images: {len(image_files)}")
print(f"YOLOv8m:      {yolo_usage_count}/{len(image_files)} ({yolo_usage_count/len(image_files)*100:.1f}%)")
print(f"Faster R-CNN: {rcnn_usage_count}/{len(image_files)} ({rcnn_usage_count/len(image_files)*100:.1f}%)")
print(f"SSD:          {ssd_usage_count}/{len(image_files)} ({ssd_usage_count/len(image_files)*100:.1f}%)")

# Save summary to Drive
os.makedirs(f"{DRIVE_DIR}/inference", exist_ok=True)
with open(f"{DRIVE_DIR}/inference/usage_summary.txt", "w") as f:
    f.write(f"Total images: {len(image_files)}\n")
    f.write(f"YOLOv8m:      {yolo_usage_count}/{len(image_files)} ({yolo_usage_count/len(image_files)*100:.1f}%)\n")
    f.write(f"Faster R-CNN: {rcnn_usage_count}/{len(image_files)} ({rcnn_usage_count/len(image_files)*100:.1f}%)\n")
    f.write(f"SSD:          {ssd_usage_count}/{len(image_files)} ({ssd_usage_count/len(image_files)*100:.1f}%)\n")

# Copy CSVs to Drive
shutil.copy(f"{OUTPUT_DIR}/yolo/results.csv",       f"{DRIVE_DIR}/inference/yolo_results.csv")
shutil.copy(f"{OUTPUT_DIR}/fasterrcnn/results.csv", f"{DRIVE_DIR}/inference/fasterrcnn_results.csv")
shutil.copy(f"{OUTPUT_DIR}/ssd/results.csv",        f"{DRIVE_DIR}/inference/ssd_results.csv")

print(f"\nAll results saved to Drive at RD2/inference/")
print("Annotated images saved locally at /content/inference_output/")
print("\nTo download annotated images, run the next cell.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda
Total images: 157

RUNNING YOLOV8 INFERENCE
YOLOv8 — Usage detected in 157/157 images

RUNNING FASTER R-CNN INFERENCE
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 98.2MB/s]


Faster R-CNN — Usage detected in 154/157 images

RUNNING SSD INFERENCE
Downloading: "https://download.pytorch.org/models/vgg16_features-amdegroot-88682ab5.pth" to /root/.cache/torch/hub/checkpoints/vgg16_features-amdegroot-88682ab5.pth


100%|██████████| 528M/528M [00:04<00:00, 126MB/s]


SSD — Usage detected in 148/157 images

SMARTPHONE USAGE DETECTION SUMMARY
Total images: 157
YOLOv8m:      157/157 (100.0%)
Faster R-CNN: 154/157 (98.1%)
SSD:          148/157 (94.3%)

All results saved to Drive at RD2/inference/
Annotated images saved locally at /content/inference_output/

To download annotated images, run the next cell.


In [5]:
import shutil

# Zip all annotated images and save to Drive
shutil.make_archive(
    f"{DRIVE_DIR}/inference/annotated_images",
    "zip",
    OUTPUT_DIR
)
print("✅ Annotated images zipped and saved to Drive at RD2/inference/annotated_images.zip")

✅ Annotated images zipped and saved to Drive at RD2/inference/annotated_images.zip


In [6]:
import numpy as np
from scipy import stats

# ─── REAL RESULTS FROM K-FOLD TRAINING ───────────────────────────────────────

# YOLOv8m - per fold mAP50, precision, recall
yolo_map50     = [0.9950, 0.9791, 0.9941, 0.9747, 0.9950]
yolo_precision = [0.9967, 0.9916, 0.9760, 0.9352, 0.9925]
yolo_recall    = [0.9913, 0.9688, 0.9991, 0.9218, 0.9994]

# Faster R-CNN - per fold precision, recall, f1
rcnn_precision = [0.8421, 0.9412, 0.8986, 0.9394, 0.9538]
rcnn_recall    = [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]
rcnn_f1        = [0.9143, 0.9697, 0.9466, 0.9688, 0.9764]

# SSD - per fold precision, recall, f1
ssd_precision  = [0.9844, 0.9846, 0.9254, 0.9524, 0.9394]
ssd_recall     = [0.9844, 1.0000, 1.0000, 0.9677, 1.0000]
ssd_f1         = [0.9844, 0.9922, 0.9612, 0.9600, 0.9688]

# Compute F1 for YOLOv8 from precision and recall
yolo_f1 = [
    2 * p * r / (p + r) if (p + r) > 0 else 0
    for p, r in zip(yolo_precision, yolo_recall)
]

# Usage detection results
usage_results = {
    "YOLOv8m":      {"detected": 157, "total": 157},
    "Faster R-CNN": {"detected": 154, "total": 157},
    "SSD":          {"detected": 148, "total": 157},
}

print("=" * 60)
print("STATISTICAL ANALYSIS REPORT")
print("=" * 60)

# ─── SECTION 1: DESCRIPTIVE STATISTICS ───────────────────────────────────────
print("\n--- 1. DESCRIPTIVE STATISTICS (Precision) ---")
for name, scores in [("YOLOv8m", yolo_precision),
                      ("Faster R-CNN", rcnn_precision),
                      ("SSD", ssd_precision)]:
    mean = np.mean(scores)
    std  = np.std(scores)
    se   = stats.sem(scores)
    ci   = stats.t.interval(0.95, df=len(scores)-1, loc=mean, scale=se)
    print(f"  {name}:")
    print(f"    Mean:  {mean:.4f}")
    print(f"    Std:   {std:.4f}")
    print(f"    95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

print("\n--- 2. DESCRIPTIVE STATISTICS (Recall) ---")
for name, scores in [("YOLOv8m", yolo_recall),
                      ("Faster R-CNN", rcnn_recall),
                      ("SSD", ssd_recall)]:
    mean = np.mean(scores)
    std  = np.std(scores)
    se   = stats.sem(scores)
    ci   = stats.t.interval(0.95, df=len(scores)-1, loc=mean, scale=se)
    print(f"  {name}:")
    print(f"    Mean:  {mean:.4f}")
    print(f"    Std:   {std:.4f}")
    print(f"    95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

print("\n--- 3. DESCRIPTIVE STATISTICS (F1 Score) ---")
for name, scores in [("YOLOv8m", yolo_f1),
                      ("Faster R-CNN", rcnn_f1),
                      ("SSD", ssd_f1)]:
    mean = np.mean(scores)
    std  = np.std(scores)
    se   = stats.sem(scores)
    ci   = stats.t.interval(0.95, df=len(scores)-1, loc=mean, scale=se)
    print(f"  {name}:")
    print(f"    Mean:  {mean:.4f}")
    print(f"    Std:   {std:.4f}")
    print(f"    95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

# ─── SECTION 2: ONE-WAY ANOVA ─────────────────────────────────────────────────
print("\n--- 4. ONE-WAY ANOVA ---")
for metric_name, a, b, c in [
    ("Precision", yolo_precision, rcnn_precision, ssd_precision),
    ("Recall",    yolo_recall,    rcnn_recall,    ssd_recall),
    ("F1 Score",  yolo_f1,        rcnn_f1,        ssd_f1),
]:
    f_stat, p_val = stats.f_oneway(a, b, c)
    sig = "✅ Significant (p < 0.05)" if p_val < 0.05 else "❌ Not significant (p >= 0.05)"
    print(f"  {metric_name}: F={f_stat:.4f}, p={p_val:.4f} → {sig}")

# ─── SECTION 3: PAIRWISE T-TESTS ─────────────────────────────────────────────
print("\n--- 5. PAIRWISE PAIRED T-TESTS (F1 Score) ---")
pairs = [
    ("YOLOv8m", "Faster R-CNN", yolo_f1, rcnn_f1),
    ("YOLOv8m", "SSD",          yolo_f1, ssd_f1),
    ("Faster R-CNN", "SSD",     rcnn_f1, ssd_f1),
]
for name1, name2, a, b in pairs:
    t_stat, p_val = stats.ttest_rel(a, b)
    sig = "✅ Significant" if p_val < 0.05 else "❌ Not significant"
    print(f"  {name1} vs {name2}: t={t_stat:.4f}, p={p_val:.4f} → {sig}")

print("\n--- 6. PAIRWISE PAIRED T-TESTS (Precision) ---")
pairs = [
    ("YOLOv8m", "Faster R-CNN", yolo_precision, rcnn_precision),
    ("YOLOv8m", "SSD",          yolo_precision, ssd_precision),
    ("Faster R-CNN", "SSD",     rcnn_precision, ssd_precision),
]
for name1, name2, a, b in pairs:
    t_stat, p_val = stats.ttest_rel(a, b)
    sig = "✅ Significant" if p_val < 0.05 else "❌ Not significant"
    print(f"  {name1} vs {name2}: t={t_stat:.4f}, p={p_val:.4f} → {sig}")

print("\n--- 7. PAIRWISE PAIRED T-TESTS (Recall) ---")
pairs = [
    ("YOLOv8m", "Faster R-CNN", yolo_recall, rcnn_recall),
    ("YOLOv8m", "SSD",          yolo_recall, ssd_recall),
    ("Faster R-CNN", "SSD",     rcnn_recall, ssd_recall),
]
for name1, name2, a, b in pairs:
    t_stat, p_val = stats.ttest_rel(a, b)
    sig = "✅ Significant" if p_val < 0.05 else "❌ Not significant"
    print(f"  {name1} vs {name2}: t={t_stat:.4f}, p={p_val:.4f} → {sig}")

# ─── SECTION 4: USAGE DETECTION ───────────────────────────────────────────────
print("\n--- 8. SMARTPHONE USAGE DETECTION ACCURACY ---")
for name, r in usage_results.items():
    pct = r["detected"] / r["total"] * 100
    print(f"  {name}: {r['detected']}/{r['total']} ({pct:.1f}%)")

# ─── SECTION 5: NORMALITY TEST ────────────────────────────────────────────────
print("\n--- 9. SHAPIRO-WILK NORMALITY TEST (F1 Scores) ---")
for name, scores in [("YOLOv8m", yolo_f1),
                      ("Faster R-CNN", rcnn_f1),
                      ("SSD", ssd_f1)]:
    stat, p_val = stats.shapiro(scores)
    normal = "Normal" if p_val > 0.05 else "Not normal"
    print(f"  {name}: W={stat:.4f}, p={p_val:.4f} → {normal} distribution")

# ─── SAVE REPORT TO DRIVE ─────────────────────────────────────────────────────
import io, sys

report_path = "/content/drive/MyDrive/RD2/inference/statistical_analysis.txt"

# Capture print output to file
old_stdout = sys.stdout
sys.stdout = buffer = io.StringIO()

# Re-run all prints into buffer
print("=" * 60)
print("STATISTICAL ANALYSIS REPORT")
print("=" * 60)
print(f"\nYOLOv8m F1 per fold:     {[round(x,4) for x in yolo_f1]}")
print(f"Faster R-CNN F1 per fold: {[round(x,4) for x in rcnn_f1]}")
print(f"SSD F1 per fold:          {[round(x,4) for x in ssd_f1]}")

for metric_name, a, b, c in [
    ("Precision", yolo_precision, rcnn_precision, ssd_precision),
    ("Recall",    yolo_recall,    rcnn_recall,    ssd_recall),
    ("F1",        yolo_f1,        rcnn_f1,        ssd_f1),
]:
    f_stat, p_val = stats.f_oneway(a, b, c)
    print(f"\nANOVA {metric_name}: F={f_stat:.4f}, p={p_val:.4f}")

for name1, name2, a, b in [
    ("YOLOv8m", "Faster R-CNN", yolo_f1, rcnn_f1),
    ("YOLOv8m", "SSD",          yolo_f1, ssd_f1),
    ("Faster R-CNN", "SSD",     rcnn_f1, ssd_f1),
]:
    t_stat, p_val = stats.ttest_rel(a, b)
    print(f"T-test F1 {name1} vs {name2}: t={t_stat:.4f}, p={p_val:.4f}")

sys.stdout = old_stdout
with open(report_path, "w") as f:
    f.write(buffer.getvalue())

print("✅ Statistical analysis saved to Drive at RD2/inference/statistical_analysis.txt")

STATISTICAL ANALYSIS REPORT

--- 1. DESCRIPTIVE STATISTICS (Precision) ---
  YOLOv8m:
    Mean:  0.9784
    Std:   0.0227
    95% CI: [0.9469, 1.0099]
  Faster R-CNN:
    Mean:  0.9150
    Std:   0.0409
    95% CI: [0.8582, 0.9718]
  SSD:
    Mean:  0.9572
    Std:   0.0238
    95% CI: [0.9241, 0.9903]

--- 2. DESCRIPTIVE STATISTICS (Recall) ---
  YOLOv8m:
    Mean:  0.9761
    Std:   0.0293
    95% CI: [0.9353, 1.0168]
  Faster R-CNN:
    Mean:  1.0000
    Std:   0.0000
    95% CI: [nan, nan]
  SSD:
    Mean:  0.9904
    Std:   0.0129
    95% CI: [0.9726, 1.0083]

--- 3. DESCRIPTIVE STATISTICS (F1 Score) ---
  YOLOv8m:
    Mean:  0.9772
    Std:   0.0250
    95% CI: [0.9425, 1.0119]
  Faster R-CNN:
    Mean:  0.9552
    Std:   0.0228
    95% CI: [0.9236, 0.9868]
  SSD:
    Mean:  0.9733
    Std:   0.0128
    95% CI: [0.9555, 0.9911]

--- 4. ONE-WAY ANOVA ---
  Precision: F=4.5296, p=0.0342 → ✅ Significant (p < 0.05)
  Recall: F=1.6943, p=0.2249 → ❌ Not significant (p >= 0.05)
  F1 Sco

/usr/local/lib/python3.12/dist-packages/scipy/stats/_distn_infrastructure.py:2323: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
/usr/local/lib/python3.12/dist-packages/scipy/stats/_distn_infrastructure.py:2324: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc
